# Montra Hospitality Intel: Data Extraction & Exploratory Data Analysis (EDA)
**Guest Experience Analytics & Topic Modelling**

This notebook extracts operational tables from the Montra PostgreSQL database hosted on NeonDB, conducts exploratory data analysis on guest bookings, rooms, and reviews, and consolidates the tables into a single unified dataset for natural language processing.

In [1]:
import os
import re
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, text

# Load environment variables securely from .env (auto-searches parent directories)
dotenv_path = find_dotenv() or os.path.abspath(os.path.join(os.getcwd(), '..', '.env')) or ".env"
load_dotenv(dotenv_path)
Database_url = os.getenv("DATABASE_URL")

if not Database_url:
    raise ValueError(
        "DATABASE_URL not found! Please set DATABASE_URL in your .env file "
        "as described in .env.example (as described in documentation)."
    )


In [2]:
#### Create database engine connection
engine = create_engine(
    Database_url,
    poolclass=None,
    connect_args={
        "sslmode": "require",
        "connect_timeout": 10
    }
)

###### Test the connection
try: 
    with engine.connect() as connection:
        result = connection.execute(text("SELECT 1"))
        print("Database response:", result.fetchone())
        print("Successful connection to Montra PostgreSQL database!")
except Exception as e:
    print("Connection failed:", repr(e))


Database response: (1,)
Successful connection to Montra PostgreSQL database!


In [3]:
# Inspect available tables in the database
query = """ 
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public';
"""

tables_df = pd.read_sql(query, engine)
tables_df


,table_name
0,bookings
1,guests
2,hotels
3,reviews
4,clean_bookings


### 1. Load Source Tables
Extract the four operational tables: `bookings`, `guests`, `hotels`, and `reviews`.

In [4]:
# 1. Load bookings data
booking = pd.read_sql("SELECT * FROM bookings", engine)
print("Bookings shape:", booking.shape)
booking.head()


Bookings shape: (35000, 9)


,booking_id,guest_id,hotel_id,room_type,booking_channel,check_in_date,check_out_date,total_amount,stay_duration
0,1,6699,18,Suite,Direct Website,2025-05-25,2025-05-26,296.44,1
1,2,11380,23,Deluxe,Corporate Portal,2025-01-09,2025-01-13,483.56,4
2,3,4812,13,Executive,Travel Agent,2025-03-04,2025-03-06,415.59,2
3,4,982,12,Family Room,Airbnb,2026-03-06,2026-03-09,601.18,3
4,5,615,5,Deluxe,Expedia,2025-02-25,2025-03-02,603.85,5


In [5]:
# 2. Load guests data
guests = pd.read_sql("SELECT * FROM guests", engine)
print("Guests shape:", guests.shape)
guests.head()


Guests shape: (12000, 5)


,guest_id,customer_type,loyalty_status,country,previous_stays
0,1,Leisure,NaN,Australia,0
1,2,Leisure,NaN,Germany,1
2,3,Business,Gold,United Kingdom,0
3,4,Family,Silver,United Arab Emirates,8
4,5,Corporate,NaN,France,0


In [6]:
# 3. Load hotels data
hotel = pd.read_sql("SELECT * FROM hotels", engine)
print("Hotels shape:", hotel.shape)
hotel.head()


Hotels shape: (25, 5)


,hotel_id,hotel_name,location,star_rating,total_rooms
0,1,The Birmingham Grand,Birmingham,4,254
1,2,The London Heritage,London,5,234
2,3,The Oxford Boutique,Oxford,3,146
3,4,The Glasgow Heritage,Glasgow,3,84
4,5,Glasgow Metropolitan,Glasgow,4,214


In [7]:
# 4. Load reviews data
review = pd.read_sql("SELECT * FROM reviews", engine)
print("Reviews shape:", review.shape)
review.head()


Reviews shape: (35000, 8)


,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,negative
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral


In [8]:
# Inspect schemas and summary info
print("=== Bookings Info ===")
booking.info()
print("\n=== Guests Info ===")
guests.info()
print("\n=== Hotels Info ===")
hotel.info()
print("\n=== Reviews Info ===")
review.info()


=== Bookings Info ===
<class 'pandas.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   booking_id       35000 non-null  int64  
 1   guest_id         35000 non-null  int64  
 2   hotel_id         35000 non-null  int64  
 3   room_type        35000 non-null  str    
 4   booking_channel  35000 non-null  str    
 5   check_in_date    35000 non-null  str    
 6   check_out_date   35000 non-null  str    
 7   total_amount     35000 non-null  float64
 8   stay_duration    35000 non-null  int64  
dtypes: float64(1), int64(4), str(4)
memory usage: 3.7 MB

=== Guests Info ===
<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   guest_id        12000 non-null  int64
 1   customer_type   12000 non-null  str  
 2   loyalty_status  5964 n

### 2. Exploratory Data Analysis (EDA)
Key operational metrics examined:
- Room type distribution & revenue by room type
- Booking channel volume & revenue generated by booking channel
- Stay duration patterns
- Rating score distributions

In [9]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Total counts of room_type
room_type = booking['room_type'].value_counts().reset_index()

### Revenue by room type
room_revenue = booking.groupby('room_type')['total_amount'].sum().reset_index()

### Booking channels distribution
channels = booking['booking_channel'].value_counts().reset_index()

### Revenue generated by booking channel
channel_revenue = booking.groupby('booking_channel')['total_amount'].sum().reset_index()

### Stay duration distribution
stay = booking['stay_duration'].value_counts().reset_index()


In [10]:
# Create comprehensive Plotly subplots overview
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        "Total Room Distribution", "Revenue Generated by Room Type",
        "Booking Channel Distribution", "Revenue Generated by Booking Channel",
        "Stay Duration Distribution", "Review Rating Distribution"
    )
)

ratings_dist = review['rating_score'].value_counts().sort_index().reset_index()

## Add charts to subplots
fig.add_trace(go.Bar(x=room_type['room_type'], y=room_type['count'], name='Count', marker_color='teal'), row=1, col=1)
fig.add_trace(go.Bar(x=room_revenue['room_type'], y=room_revenue['total_amount'], name='Revenue', marker_color='indianred'), row=1, col=2)
fig.add_trace(go.Bar(x=channels['booking_channel'], y=channels['count'], name='Count', marker_color='blue'), row=2, col=1)
fig.add_trace(go.Bar(x=channel_revenue['booking_channel'], y=channel_revenue['total_amount'], name='Revenue', marker_color='yellow'), row=2, col=2)
fig.add_trace(go.Bar(x=stay['stay_duration'], y=stay['count'], name='Count', marker_color='goldenrod'), row=3, col=1)
fig.add_trace(go.Bar(x=ratings_dist['rating_score'], y=ratings_dist['count'], name='Ratings', marker_color='mediumseagreen'), row=3, col=2)

fig.update_layout(height=900, title_text="Montra Hospitality Operational & Review Analysis Overview", showlegend=False)
fig.show()


### 3. Data Integration & Consolidation
The four operational tables are joined using relational inner joins:
- `reviews` joins with `bookings` on `['booking_id', 'guest_id', 'hotel_id']`
- merged joins with `guests` on `'guest_id'`
- merged joins with `hotels` on `'hotel_id'`

In [11]:
# Perform relational merges across all 4 tables
merge = review.merge(booking, on=['booking_id', 'guest_id', 'hotel_id'], how="inner")
merge = merge.merge(guests, on='guest_id', how='inner')
merge = merge.merge(hotel, on='hotel_id', how="inner")

print("Merged dataset shape:", merge.shape)
merge.head()


Merged dataset shape: (35000, 22)


,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,room_type,booking_channel,...,total_amount,stay_duration,customer_type,loyalty_status,country,previous_stays,hotel_name,location,star_rating,total_rooms
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,Suite,Direct Website,...,296.44,1,Leisure,NaN,France,4,The Aberdeen Grand,Aberdeen,4,118
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive,Deluxe,Corporate Portal,...,483.56,4,Business,Gold,Germany,4,Nottingham Park Hotel,Nottingham,4,95
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative,Executive,Travel Agent,...,415.59,2,Business,NaN,Ireland,4,The London Boutique,London,4,231
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,negative,Family Room,Airbnb,...,601.18,3,Leisure,NaN,Nigeria,0,Manchester Metropolitan,Manchester,3,101
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral,Deluxe,Expedia,...,603.85,5,Group,NaN,United States,9,Glasgow Metropolitan,Glasgow,4,214


In [12]:
# Check missing values in the merged dataset
print("Missing values per column:")
print(merge.isna().sum())


Missing values per column:
review_id              0
booking_id             0
guest_id               0
hotel_id               0
review_text            0
rating_score           0
review_date            0
sentiment_label        0
room_type              0
booking_channel        0
check_in_date          0
check_out_date         0
total_amount           0
stay_duration          0
customer_type          0
loyalty_status     17615
country                0
previous_stays         0
hotel_name             0
location               0
star_rating            0
total_rooms            0
dtype: int64


### 4. Save the Consolidated Dataset
Save the consolidated dataset to `../data/processed/merged_raw.csv` using clean relative paths.

In [13]:
# Save consolidated data
os.makedirs("../data/processed", exist_ok=True)
output_path = "../data/processed/merged_raw.csv"
merge.to_csv(output_path, index=False)
print(f"Successfully saved {merge.shape[0]} records to {output_path}")


Successfully saved 35000 records to ../data/processed/merged_raw.csv
